# Reinforcement Learning: SARSA in CliffWalking-v1

This notebook demonstrates the implementation of the SARSA (State-Action-Reward-State-Action) reinforcement learning algorithm to solve the `CliffWalking-v1` environment from the Gymnasium library.

## 1. Environment Setup and Library Imports

First, we install and import the necessary libraries, including `gymnasium` for the environment, `numpy` for numerical operations, and `random` for implementing the epsilon-greedy policy.

### Gymnasium Environment: CliffWalking-v1

The `CliffWalking-v1` environment is a classic reinforcement learning problem where an agent must navigate from a starting point to a goal without falling off a 'cliff'.

- **Observation Space**: The environment has 48 possible states, representing the agent's position on a 4x12 grid.
- **Action Space**: The agent can take 4 discrete actions: Up, Right, Down, Left.

`env.reset()` initializes the environment and returns the initial state and an info dictionary. The initial state is typically state 36 in this environment.

In [ ]:
!pip install gymnasium

In [ ]:
!pip install "gymnasium[toy-text]"

In [ ]:
import gymnasium as gym
import numpy as np
import random

In [ ]:
env = gym.make("CliffWalking-v1")

In [ ]:
print(env.observation_space.n) #states
print(env.action_space.n) #actions
starting_state,_ = env.reset() #36 = 3*12+0
print(starting_state) #36

48
4
36


## SARSA

## 2. SARSA Algorithm Parameters

Before training, we define the hyperparameters for the SARSA algorithm:

- `gamma` (Discount Factor): Determines the importance of future rewards. A value closer to 1 makes the agent consider future rewards more heavily.
- `alpha` (Learning Rate): Controls how much new information overrides old information. A higher value means the agent learns faster.
- `epsilon` (Exploration Rate): Dictates the probability of taking a random action (exploration) instead of the action with the highest Q-value (exploitation). It encourages the agent to discover new paths.
- `episodes`: The total number of training iterations.

In [ ]:
gamma = 0.99
alpha = 0.5
epsilon = 0.1
episodes = 500

## 3. Q-Table Initialization

The Q-table is the core data structure in Q-learning and SARSA. It stores the estimated maximum future reward for taking an action in a given state.

- It's initialized as a NumPy array of zeros with dimensions `(number_of_states, number_of_actions)`.
- Each `Q[s, a]` entry represents the quality of taking action `a` in state `s`.

In [ ]:
# Q-table => store Q-values
Q = np.zeros((48, 4))

## 4. Epsilon-Greedy Policy

This policy balances exploration (trying new actions) and exploitation (using known best actions):

- With probability `epsilon`, the agent chooses a random action (`env.action_space.sample()`). This helps discover potentially better paths.
- With probability `1 - epsilon`, the agent chooses the action with the highest Q-value for the current state (`np.argmax(Q[state])`). This leverages the agent's current knowledge.

In [ ]:
#Policy - epsilon-greedy : state->action
def epsilon_greedy(state):
    if random.random() < epsilon:
        return env.action_space.sample() #random action => EXPLORE
    else:
        return np.argmax(Q[state]) #exploit

## 5. SARSA Training Loop

The agent trains over a specified number of `episodes`. In each episode:

1.  **Environment Reset**: The environment is reset, and the initial state is obtained.
2.  **Initial Action Selection**: An action is chosen using the `epsilon_greedy` policy based on the initial state.
3.  **Step through Environment**: The agent takes the chosen action, and the environment returns the `next_state`, `reward`, and `done` flags (`terminated` or `truncated`).
4.  **Next Action Selection**: Crucially for SARSA, the *next action* (`next_action`) is chosen from the `next_state` using the `epsilon_greedy` policy *before* updating the Q-value.
5.  **SARSA Update**: The Q-value for the `(state, action)` pair is updated using the SARSA formula:
    $$Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma Q(s', a') - Q(s, a)]$$
    Where:
    -   `s`: current state
    -   `a`: current action
    -   `r`: reward received
    -   `s'`: next state
    -   `a'`: next action (chosen using the same policy in `s'`)
6.  **State and Action Update**: The `state` and `action` are updated to `next_state` and `next_action` respectively.
7.  **Episode Termination**: The loop continues until `done` is `True`.
8.  **Rendering**: For visualization, the environment is rendered every 50 episodes using `render_mode="human"` to show the agent's progress.

In [ ]:
for episode in range(episodes):
    render = (episode % 50 ==0)
    if render:
        env=gym.make("CliffWalking-v1" , render_mode="human")
        print(f"rendering env for episode={episode+1}/500")
    else:
        env=gym.make("CliffWalking-v1")
    done = False #when the epiosde has ended
    state,_ = env.reset()
    action = epsilon_greedy(state)
    total_reward = 0
    episode_len = 0
    while not done:
        next_state , reward , terminated , truncated,_ = env.step(action)
        done = terminated or truncated
        next_action = epsilon_greedy(next_state)
        #SARSA update
        Q[state, action] += alpha * (reward + gamma*(Q[next_state, next_action] - Q[state, action]))
        state = next_state
        action = next_action
        total_reward += reward
        episode_len +=1
    print(f"epsilon={episode+1}/500: total reward = {total_reward} & ep length = {episode_len}")
    env.close()

rendering env for episode=1/500
epsilon=1/500: total reward = -105 & ep length = 105
epsilon=2/500: total reward = -2310 & ep length = 627
epsilon=3/500: total reward = -200 & ep length = 101
epsilon=4/500: total reward = -70 & ep length = 70
epsilon=5/500: total reward = -125 & ep length = 125
epsilon=6/500: total reward = -58 & ep length = 58
epsilon=7/500: total reward = -215 & ep length = 116
epsilon=8/500: total reward = -81 & ep length = 81
epsilon=9/500: total reward = -70 & ep length = 70
epsilon=10/500: total reward = -67 & ep length = 67
epsilon=11/500: total reward = -144 & ep length = 45
epsilon=12/500: total reward = -68 & ep length = 68
epsilon=13/500: total reward = -230 & ep length = 131
epsilon=14/500: total reward = -33 & ep length = 33
epsilon=15/500: total reward = -264 & ep length = 66
epsilon=16/500: total reward = -54 & ep length = 54
epsilon=17/500: total reward = -40 & ep length = 40
epsilon=18/500: total reward = -31 & ep length = 31
epsilon=19/500: total rewa

In [ ]:
# what did our agent learn ?
env= gym.make("CliffWalking-v1" , render_mode = "human")
state,_ = env.reset()
done = False
total_reward = 0
episode_len =0
while not done:
    action = np.argmax(Q[state])
    state,reward,terminated,truncated,_=env.step(action)
    done = terminated or truncated
    total_reward += reward
    episode_len += 1
print(f"total reward = {total_reward} & episode len = {episode_len}")
env.close()

total reward = -19 & episode len = 19


## 6. Evaluation of the Trained Agent

After training, we evaluate the agent's learned policy. During evaluation:

- **No Exploration**: The `epsilon_greedy` policy is replaced with a purely greedy policy (`np.argmax(Q[state])`), meaning the agent always chooses the action with the highest Q-value, demonstrating its learned behavior.
- **Rendering**: The environment is rendered to visualize the agent's optimal path.
- **Metrics**: The `total_reward` and `episode_len` are printed to assess the agent's performance (a higher reward and shorter episode length indicate better performance).

The output `total reward = -19 & episode len = 19` indicates that the agent successfully reached the goal state in 19 steps with a total reward of -19. This is the optimal path for `CliffWalking-v1` (moving right until the last column, then down to the goal), where each step incurs a reward of -1.